# Fit con alpha libero sui DATI REALI (DR2low)

Primo fit in cui l indice cromatico dello scattering e un parametro libero **sui dati reali** invece che su una simulazione.

Finora:

| | dati reali | simulazioni |
|---|---|---|
| alpha fisso (4.0, 4.4) | notebook 01, 02, 10 | notebook 07, 09, 12 |
| **alpha libero** | **questo notebook** | notebook 08, 13, 14 |

Sulle simulazioni (notebook 08) alpha libero era una validazione del metodo: alpha = 4.4 iniettato, 4.68 recuperato. Qui non c e nessun valore vero da controllare: e una misura.

Modello: 7 parametri = red_noise (2) + dm_gp (2, alpha = 2 fisso) + sv_gp (2) + `sv_gp_alpha`, tramite `ds.dmfourierbasis_alpha` con **fref = 1400 MHz** (il default del pacchetto, scelta deliberata per restare confrontabili con i notebook 08/13/14).

La catena completa (10^6 iterazioni) e prodotta su mithrandir con:

```
python run_08_alpha_libero.py --data ./dati/DR2low_feathers/J1640+2224 --outdir ./test_real_alphafree/
```

e poi copiata in locale come `test_real_alphafree_ws/`. Nessun sampling attivo in questo notebook.

Domande:
1. Quanto vale alpha nei dati reali, e la posteriore e informativa o riproduce il prior?
2. Cambiano i parametri di DM e red noise rispetto al fit con alpha fisso a 4.4 (notebook 10)?
3. La degenerazione alpha - log10_A ha la pendenza prevista da nu_eff dei dati reali?
4. I dati reali si comportano come la simulazione con alpha = 4.4, o diversamente?


## Setup comune

Copia delle celle 0, 1, 3, 4, 6, 7, 9 di `Likelihood.ipynb`. `psr` e la pulsar **reale** J1640+2224 (DR2low) e `logL.params` (6 parametri) serve da riferimento per l ordine delle colonne delle catene a 6 parametri.


In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee

In [ ]:
# path to the folder with the data
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9] # we are only analysing pulsar #9

In [ ]:
# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.0, tndm = False)))

In [ ]:
logL = ds.PulsarLikelihood(model).logL

In [ ]:
# annoying function needed because the function logL want a dictionary as input, but our sampler wants a likelihood with an array as input
# so we have to switch from one to another :(
def params_dict_from_array(x):
    # ensure x is a plain numpy array before slicing
    x = np.asarray(x)
    return dict(zip(logL.params, x))

In [ ]:
##### Cell for the prior distributions ###################################################################################

# dictionary with minimum and maximum value of the prior distribution
my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo) ## this is where the uniform distribution is defined
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

priors = build_priors(logL.params, my_priordict)
priors

def log_prior(x):
    """Compute log prior"""
    params = params_dict_from_array(x)

    logp = 0.0
    for name, value in params.items():
        if name in priors:
            logp += priors[name].logpdf(value)
        else:
            # Default: standard normal prior
            if isinstance(value, np.ndarray):
                logp += -0.5 * np.sum(value**2)
            else:
                logp += -0.5 * value**2

    return logp

def prior_draw():
    """Draw from prior and return as flat array"""
    x = []
    for name in logL.params:
        x.append(priors[name].rvs())
    return np.array(x)

x0 = prior_draw()
x0

In [ ]:
@jax.jit # compile the likelihood in jax to make it faster
def _jit_logL(params_array):
    params = dict(zip(logL.params, params_array))
    return logL(params)

_jit_logL(x0)

def log_likelihood(x):
    """
    Likelihood function that is given to the samples, it uses the jitted (=faster) version of the likelihood
    """
    x = np.asarray(x, dtype=np.float64)
    result = _jit_logL(jnp.array(x))
    return float(np.array(result))  # always a plain Python float

## Modello con alpha libero sui dati reali

Stessa pulsar `psr` del setup comune (nessun nuovo caricamento: i dati reali sono gia quelli). Cambia solo la base dello scattering: `ds.dmfourierbasis_alpha` invece di `ds.make_dmfourierbasis(alpha=4.0)`, quindi alpha diventa il settimo parametro campionato.


In [ ]:
# stesso modello a 6 par sui dati REALI, ma lo scattering usa dmfourierbasis_alpha: alpha diventa un parametro libero
model_real_af = [psr.residuals]
model_real_af += [ds.makegp_timing(psr, svd=True)]
model_real_af += [ds.makenoise_measurement(psr, psr.noisedict, tnequad=True, ecorr=True)]
model_real_af.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'],
                                       T=ds.getspan(psr), name='red_noise'))
model_real_af.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'],
                                       T=ds.getspan(psr), name='dm_gp',
                                       fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_real_af.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'],
                                       T=ds.getspan(psr), name='sv_gp',
                                       fourierbasis=ds.dmfourierbasis_alpha))   # <-- alpha libero, fref = 1400 (default)

logL_real_af = ds.PulsarLikelihood(model_real_af).logL
logL_real_af.params   # attesi 7 parametri, compreso J1640+2224_sv_gp_alpha

In [ ]:
# controllo: i 6 parametri standard + sv_gp_alpha
atteso = set(logL.params) | {'J1640+2224_sv_gp_alpha'}
assert set(logL_real_af.params) == atteso, \
    f"Mancanti: {atteso - set(logL_real_af.params)}\nIn eccesso: {set(logL_real_af.params) - atteso}"
print('OK: 7 parametri attesi')
print('ATTENZIONE: ordine diverso da logL.params ->', logL_real_af.params)

In [ ]:
def params_dict_from_array_real_af(x):
    x = np.asarray(x)
    return dict(zip(logL_real_af.params, x))

priors_real_af = build_priors(logL_real_af.params, my_priordict)   # sv_gp_alpha: uniforme [0, 10]
print('prior su alpha:', priors_real_af['J1640+2224_sv_gp_alpha'].support())

def log_prior_real_af(x):
    params = params_dict_from_array_real_af(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_real_af:
            logp += priors_real_af[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_real_af():
    return np.array([priors_real_af[name].rvs() for name in logL_real_af.params])

x0_real_af = prior_draw_real_af()
print('x0 =', x0_real_af)

In [ ]:
@jax.jit
def _jit_logL_real_af(params_array):
    return logL_real_af(dict(zip(logL_real_af.params, params_array)))

def log_likelihood_real_af(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_real_af(jnp.array(x))))

print('logL iniziale =', log_likelihood_real_af(x0_real_af))

# tempo di una valutazione (dopo la compilazione): serve a stimare la durata del sampling
start = time.time()
for _ in range(20):
    log_likelihood_real_af(x0_real_af)
dt = (time.time() - start) / 20
print(f'una valutazione = {dt*1e3:.1f} ms  ->  10^6 iterazioni ~ {dt*1e6/3600:.1f} ore (solo likelihood)')

In [ ]:
sampler_real_af = ptmcmc(
    ndim=len(logL_real_af.params),
    logl=log_likelihood_real_af,
    logp=log_prior_real_af,
    cov=np.eye(len(logL_real_af.params)) * 0.001,
    outDir='./test_real_alphafree/',
    resume=True
)

### Sampling (lungo)

**La run vera va fatta su mithrandir**, non qui: `run_08_alpha_libero.py --data ./dati/DR2low_feathers/J1640+2224 --outdir ./test_real_alphafree/`.

Questa cella resta solo per poter eventualmente fare una catena parziale di prova sul Mac. Lasciare `ESEGUI_SAMPLING = False`.


In [ ]:
ESEGUI_SAMPLING = False   # True solo per (ri)lanciare davvero il campionamento (ore)

if ESEGUI_SAMPLING:
    print("Starting sampling (DATI REALI DR2low, analizzati con alpha LIBERO)...")
    sampler_real_af.sample(x0_real_af, Niter=1000000, thin=10)
else:
    print('sampling saltato: la catena completa e in test_real_alphafree_ws/')

## Catena e diagnostiche

**Attenzione all ordine delle colonne**: qui segue `logL_real_af.params` (7 parametri, `sv_gp_alpha` compreso), non `logL.params`. Per i confronti con le catene a 6 parametri le colonne sono selezionate **per nome**, mai per posizione.


In [ ]:
# catena COMPLETA (10^6 iterazioni) prodotta su mithrandir con run_08_alpha_libero.py;
# per usare una eventuale catena parziale fatta sul Mac, togliere il suffisso _ws dal percorso
chain_path_real_af = './test_real_alphafree_ws/chain_1.txt'

if not os.path.exists(chain_path_real_af):
    raise FileNotFoundError(
        f'catena non trovata: {chain_path_real_af}\n'
        'La run non e ancora stata fatta o non e ancora stata copiata da mithrandir.\n'
        'Sulla workstation:  python run_08_alpha_libero.py '
        '--data ./dati/DR2low_feathers/J1640+2224 --outdir ./test_real_alphafree/\n'
        'Poi in locale:      rsync -av <utente>@149.132.54.143:<path>/test_real_alphafree/ ./test_real_alphafree_ws/'
    )

chain_real_af = np.loadtxt(chain_path_real_af)
n_tot_real_af = len(chain_real_af)
chain_real_af = chain_real_af[n_tot_real_af//4:, :-4]   # 25% di burn-in, via le 4 colonne diagnostiche
print(f'righe nel file: {n_tot_real_af}  ->  campioni usati: {len(chain_real_af)}  ({100*n_tot_real_af/100000:.1f}% della run completa)')
print('shape:', chain_real_af.shape, '  atteso (~75000, 7)')

# indice di ogni parametro nella catena con alpha libero
idx_af = {name: i for i, name in enumerate(logL_real_af.params)}
idx_af

In [ ]:
tau_real_af = emcee.autocorr.integrated_time(chain_real_af, has_walkers=False, quiet=True)
ess_real_af = int(len(chain_real_af) / tau_real_af.max())
print('tau =', dict(zip([p.replace('J1640+2224_', '') for p in logL_real_af.params], np.round(tau_real_af, 1))))
print('Effective sample size =', ess_real_af)
if ess_real_af < 1000:
    print('ATTENZIONE: ESS basso, le code della posteriore non sono affidabili')

In [ ]:
# riepilogo numerico: nessun valore vero, questi sono DATI REALI
print(f"{'parametro':22s} {'mediana':>9s}  {'[16, 84]':>20s}   {'[5, 95]':>20s}")
for name in logL_real_af.params:
    col = chain_real_af[:, idx_af[name]]
    q05, q16, q50, q84, q95 = np.percentile(col, [5, 16, 50, 84, 95])
    print(f"{name.replace('J1640+2224_', ''):22s} {q50:9.3f}  [{q16:8.3f}, {q84:8.3f}]   [{q05:8.3f}, {q95:8.3f}]")

# quanto si avvicina al bordo del prior l ampiezza dello scattering?
lA = chain_real_af[:, idx_af['J1640+2224_sv_gp_log10_A']]
frac_bordo = float(np.mean(lA < -19.0))
print(f"\nfrazione di campioni con sv_gp_log10_A < -19 (ultimo dex del prior): {100*frac_bordo:.2f}%")
print('se non trascurabile, lo scattering non e rilevato e la posteriore di alpha va letta con cautela')

### Corner a 7 parametri


In [ ]:
import corner

clean_labels_af = [name.replace('J1640+2224_', '') for name in logL_real_af.params]

fig = corner.corner(
    chain_real_af,
    labels=clean_labels_af,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#264653',
)
fig.set_size_inches(18, 18)
fig.suptitle('J1640+2224 — DATI REALI DR2low, analisi con alpha LIBERO', fontsize=15, y=1.00)
fig.savefig('corner_reali_alphafree.png', dpi=150, bbox_inches='tight')

### La posteriore di alpha e informativa?

Il test diretto: sovrapporre la posteriore marginale di alpha al suo prior, che e uniforme su [0, 10]. Se le due curve coincidono, i dati non dicono niente su alpha e il numero che si legge nel titolo del corner e solo il prior. Se la posteriore e piu stretta, la differenza e informazione vera.

Il confronto quantitativo e la larghezza dell intervallo al 68% rapportata ai 10 dex di prior.


In [ ]:
a_s = chain_real_af[:, idx_af['J1640+2224_sv_gp_alpha']]
q16, q50, q84 = np.percentile(a_s, [16, 50, 84])
largh = q84 - q16
prior_lo, prior_hi = 0.0, 10.0

print(f'alpha = {q50:.2f}  68% CI = [{q16:.2f}, {q84:.2f}]  larghezza {largh:.2f}')
print(f'il prior e largo {prior_hi - prior_lo:.0f}; un prior uniforme ha 68% CI largo {0.68*(prior_hi-prior_lo):.2f}')
print(f'riduzione rispetto al prior: {100*(1 - largh/(0.68*(prior_hi-prior_lo))):.0f}%')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(a_s, bins=60, range=(prior_lo, prior_hi), density=True, color='#264653',
        alpha=0.85, label='posteriore (dati reali)')
ax.axhline(1.0/(prior_hi - prior_lo), color='#E76F51', ls='--', lw=2,
           label='prior uniforme [0, 10]')
ax.axvline(4.0, color='#999999', ls=':', lw=1.5, label='alpha = 4 (scattering)')
ax.axvline(2.0, color='#666666', ls=':', lw=1.5, label='alpha = 2 (DM)')
ax.set_xlabel('sv_gp_alpha', fontsize=13)
ax.set_ylabel('densita', fontsize=13)
ax.set_title('J1640+2224 reali — posteriore di alpha contro il suo prior', fontsize=13)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig('posterior_alpha_reali_vs_prior.png', dpi=150, bbox_inches='tight')

### Degenerazione alpha - log10_A dello scattering

Il segnale di scattering in un TOA a frequenza nu e proporzionale a A x (1400/nu)^alpha. Se i dati sono concentrati attorno a una frequenza efficace nu_eff, due coppie (alpha, log10_A) danno quasi lo stesso segnale quando

log10_A + alpha x log10(1400/nu_eff) = costante,

cioe la posteriore si allunga lungo una retta di pendenza **-log10(1400/nu_eff)**. nu_eff e stimata pesando i TOA con il loro contributo al segnale di scattering, (1400/nu)^(2 alpha) / sigma^2.

Sui dati reali non c e un punto vero a cui ancorare la retta: la ancoriamo alla mediana della posteriore.


In [ ]:
f_toa = np.asarray(psr.freqs)
sig   = np.asarray(psr.toaerrs)
a_med = float(np.median(a_s))

for a_w, etichetta in [(a_med, 'alpha mediano della posteriore'), (4.4, 'alpha = 4.4 di riferimento')]:
    w = (1400.0 / f_toa)**(2 * a_w) / sig**2
    sl = -np.sum(w * np.log10(1400.0 / f_toa)) / np.sum(w)
    print(f'{etichetta:34s} (alpha={a_w:5.2f}):  nu_eff ~ {1400.0/10**(-sl):6.0f} MHz,  pendenza attesa {sl:6.2f}')

w = (1400.0 / f_toa)**(2 * a_med) / sig**2
slope = -np.sum(w * np.log10(1400.0 / f_toa)) / np.sum(w)
nu_eff = 1400.0 / 10**(-slope)

lA_s = chain_real_af[:, idx_af['J1640+2224_sv_gp_log10_A']]
print(f'\ncorrelazione di Pearson alpha - log10_A = {np.corrcoef(a_s, lA_s)[0, 1]:.3f}')

# pendenza misurata sulla catena (regressione lineare), da confrontare con slope
slope_mis = np.polyfit(a_s, lA_s, 1)[0]
print(f'pendenza misurata sulla catena         = {slope_mis:.2f}')
print(f'pendenza prevista da nu_eff ~ {nu_eff:.0f} MHz   = {slope:.2f}')

fig = corner.corner(
    np.column_stack([a_s, lA_s]),
    labels=['sv_gp_alpha', 'sv_gp_log10_A'],
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#264653',
)
ax = fig.axes[2]   # pannello 2D in basso a sinistra
xx = np.linspace(a_s.min(), a_s.max(), 100)
lA_med = float(np.median(lA_s))
ax.plot(xx, lA_med + slope * (xx - a_med), 'r--', lw=1.5,
        label=f'attesa da nu_eff={nu_eff:.0f} MHz (pendenza {slope:.2f})')
ax.legend(fontsize=9)
fig.set_size_inches(8, 8)
fig.suptitle('J1640+2224 reali — degenerazione alpha / ampiezza dello scattering', fontsize=12, y=1.00)
fig.savefig('corner_reali_alphafree_degenerazione.png', dpi=150, bbox_inches='tight')

### Confronto sui 6 parametri comuni: alpha fisso a 4.4 (notebook 10) vs alpha libero

La domanda e se liberare alpha sposta le conclusioni su DM e red noise, cioe se il valore di alpha che si assume cambia la fisica che si deduce. Le colonne della catena con alpha libero sono riordinate **per nome** secondo `logL.params`, cosi i pannelli corrispondono.

Il fit con alpha fisso e disegnato per primo (arancio), quello con alpha libero sopra (blu), per non coprirlo.


In [ ]:
path_a44 = './test_real_sv44/chain_1.txt'

if not os.path.exists(path_a44):
    print('catena con alpha fisso assente, confronto saltato:', path_a44)
else:
    chain_a44 = np.loadtxt(path_a44)
    n_a44 = len(chain_a44)
    chain_a44 = chain_a44[n_a44//4:, :-4]
    print(f'alpha fisso 4.4: {len(chain_a44)} campioni, shape {chain_a44.shape} (atteso 6 colonne)')

    # colonne della catena alpha-libero riordinate come logL.params (6 parametri, alpha escluso)
    chain_af_6 = chain_real_af[:, [idx_af[name] for name in logL.params]]
    clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

    print(f"\n{'parametro':22s} {'alpha=4.4':>22s} | {'alpha libero':>22s} | {'shift':>7s}")
    for i, name in enumerate(logL.params):
        a16, a50, a84 = np.percentile(chain_a44[:, i], [16, 50, 84])
        b16, b50, b84 = np.percentile(chain_af_6[:, i], [16, 50, 84])
        sigma = 0.5 * ((a84 - a16) + (b84 - b16)) / 2.0
        shift = (b50 - a50) / sigma if sigma > 0 else np.nan
        print(f"{name.replace('J1640+2224_', ''):22s} "
              f"{a50:8.3f} [{a16:6.2f},{a84:6.2f}] | "
              f"{b50:8.3f} [{b16:6.2f},{b84:6.2f}] | {shift:6.2f}s")

    fig = corner.corner(
        chain_a44, labels=clean_labels, show_titles=True, title_fmt='.2f',
        label_kwargs={'fontsize': 14}, title_kwargs={'fontsize': 11},
        hist_kwargs={'density': True}, fill_contours=True,
        levels=(0.68, 0.95, 0.997), smooth=1.0, bins=40, color='#E76F51',
    )
    corner.corner(
        chain_af_6, labels=clean_labels, show_titles=True, title_fmt='.2f',
        label_kwargs={'fontsize': 14}, title_kwargs={'fontsize': 11},
        hist_kwargs={'density': True}, fill_contours=True,
        levels=(0.68, 0.95, 0.997), smooth=1.0, bins=40, color='#264653', fig=fig,
    )
    fig.set_size_inches(16, 16)
    fig.suptitle('J1640+2224 reali — alpha fisso a 4.4 (arancio) vs alpha libero (blu)', fontsize=13, y=1.00)
    fig.savefig('corner_reali_alpha44_vs_alphafree.png', dpi=150, bbox_inches='tight')

### Confronto dati reali / simulazione, entrambi con alpha libero

La simulazione di riferimento e `test_sim_sv44_alphafree_ws/` (notebook 08): scattering iniettato con alpha = 4.4, nessun GWB, analizzata con alpha libero. E il caso in cui sappiamo che il modello e quello giusto, quindi serve da metro per capire se i dati reali si comportano in modo anomalo.

La catena simulata viene dallo stesso modello a 7 parametri sulla stessa pulsar, quindi l ordine dei parametri coincide con `logL_real_af.params`: riuso `idx_af` anche per lei. Il controllo sul numero di colonne lo verifica.


In [ ]:
path_sim = './test_sim_sv44_alphafree_ws/chain_1.txt'

if not os.path.exists(path_sim):
    print('baseline simulata assente, confronto saltato:', path_sim)
else:
    chain_sim = np.loadtxt(path_sim)
    n_sim = len(chain_sim)
    chain_sim = chain_sim[n_sim//4:, :-4]
    assert chain_sim.shape[1] == len(logL_real_af.params), \
        f'colonne inattese nella catena simulata: {chain_sim.shape[1]} invece di {len(logL_real_af.params)}'
    print(f'simulazione alpha libero: {len(chain_sim)} campioni, shape {chain_sim.shape}')

    print(f"\n{'parametro':22s} {'simulazione':>22s} | {'dati reali':>22s}")
    for name in logL_real_af.params:
        s16, s50, s84 = np.percentile(chain_sim[:, idx_af[name]], [16, 50, 84])
        r16, r50, r84 = np.percentile(chain_real_af[:, idx_af[name]], [16, 50, 84])
        print(f"{name.replace('J1640+2224_', ''):22s} "
              f"{s50:8.3f} [{s16:6.2f},{s84:6.2f}] | "
              f"{r50:8.3f} [{r16:6.2f},{r84:6.2f}]")

    fig = corner.corner(
        chain_sim, labels=clean_labels_af, show_titles=True, title_fmt='.2f',
        label_kwargs={'fontsize': 14}, title_kwargs={'fontsize': 11},
        hist_kwargs={'density': True}, fill_contours=True,
        levels=(0.68, 0.95, 0.997), smooth=1.0, bins=40, color='#2A9D8F',
    )
    corner.corner(
        chain_real_af, labels=clean_labels_af, show_titles=True, title_fmt='.2f',
        label_kwargs={'fontsize': 14}, title_kwargs={'fontsize': 11},
        hist_kwargs={'density': True}, fill_contours=True,
        levels=(0.68, 0.95, 0.997), smooth=1.0, bins=40, color='#264653', fig=fig,
    )
    fig.set_size_inches(18, 18)
    fig.suptitle('alpha libero — simulazione con alpha=4.4 iniettato (verde) vs DATI REALI (blu)',
                 fontsize=14, y=1.00)
    fig.savefig('corner_reali_vs_sim_alphafree.png', dpi=150, bbox_inches='tight')

## Cosa guardare nei risultati

1. **Posteriore di alpha contro il prior.** Se la riduzione rispetto al prior e piccola, la conclusione da scrivere in tesi e che i dati EPTA DR2low di J1640+2224 non determinano l indice cromatico, e la figura `posterior_alpha_reali_vs_prior.png` lo dimostra.
2. **Pendenza della degenerazione.** La pendenza misurata sulla catena va confrontata con quella prevista da nu_eff. Se coincidono, la degenerazione e interamente spiegata dalla copertura in frequenza del dataset: e un risultato quantitativo, non un difetto del sampler.
3. **Shift sui 6 parametri comuni.** Colonna `shift` del confronto con alpha = 4.4: se DM e red noise si spostano di piu di circa 1 sigma, assumere alpha = 4.4 non e innocuo e va dichiarato.
4. **Coda a bassa ampiezza dello scattering.** Nel fit con alpha fisso a 4.4 il 5-esimo percentile di `sv_gp_log10_A` era -18.26, cioe vicino al fondo del prior: lo scattering e rilevato debolmente. Con alpha libero la coda puo peggiorare, e in quel caso la posteriore di alpha e per costruzione poco informativa.
5. **Reali contro simulazione.** La simulazione e il caso in cui il modello e corretto per costruzione. Se nei dati reali la posteriore di alpha e molto piu larga, oppure la degenerazione ha una forma diversa, la differenza non viene dal metodo ma dai dati.
